# Lesson 7: Distance Measures

"Distance" on a pixel grid is not as simple as it sounds. We look at three different flavors:

1. **Point-to-point distance metrics** (Manhattan, chessboard, and Euclidean) &mdash; which measure how far apart two pixels are.
2. **Curve-length estimators** (Freeman and a refined weighted estimator) &mdash; which measure the length of a *digitized curve* (e.g. the boundary of a blob) from its chain code.
3. **Chamfer distance transform** &mdash; a fast, image-wide approximation to the Euclidean distance transform, computed with two raster-scan passes.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Part 1: Point-to-point distance metrics

Given two pixels $p=(x_1,y_1)$ and $q=(x_2,y_2)$, let $d_x = x_2-x_1$ and $d_y=y_2-y_1$. Three common metrics:

- **Manhattan / city-block ($L_1$)**: distance if you can move only along the grid axes (like walking city blocks).

    $D_4(p,q) = |d_x| + |d_y|$ 
- **Chessboard ($L_\infty$)**: number of king moves on a chessboard (diagonal steps are "free").

    $D_8(p,q) = \max(|d_x|,\,|d_y|)$ 
- **Euclidean ($L_2$)**: ordinary straight-line distance.

    $D_E(p,q) = \sqrt{d_x^2+d_y^2}$

The names $D_4$ and $D_8$ come from the fact that each is the shortest path length when only 4-connected (axis) or 8-connected (axis + diagonal) moves are allowed, one unit per move.

In [ ]:
def manhattan(dx, dy):
    return np.abs(dx) + np.abs(dy)

def chessboard(dx, dy):
    return np.maximum(np.abs(dx), np.abs(dy))

def euclidean(dx, dy):
    return np.sqrt(dx**2 + dy**2)

### Visualizing the three metrics as distance fields

For every pixel in a grid, we compute its distance to the center pixel under each metric and display it as an image. The metrics agree only along the axes — everywhere else they diverge.

In [ ]:
size = 101
half = size // 2
yy, xx = np.mgrid[-half:half + 1, -half:half + 1]

fields = {
    'Manhattan ($D_4$)': manhattan(xx, yy),
    'Chessboard ($D_8$)': chessboard(xx, yy),
    'Euclidean': euclidean(xx, yy),
}

# Compute a single vmin/vmax to prevent each imshow() from auto-scaling 
# (Manhattan's max is 100, Euclidean's is ~71, chessboard's is 50).
vmax = max(field.max() for field in fields.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, field) in zip(axes, fields.items()):
    im = ax.imshow(field, cmap='viridis', vmin=0, vmax=vmax)
    ax.contour(field, levels=8, colors='white', linewidths=0.5)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

The contour lines reveal each metric's characteristic shape: Manhattan distance forms diamonds, chessboard distance forms squares, and Euclidean distance forms circles — the one shape most people intuitively think of as "equal distance away".

## Part 2: Estimating curve length from a chain code

Now suppose we want to estimate the length of a *curve*, not just the distance between two points. Let's create a test image of a circle with known radius ($60$) and known circumference ($120 \pi = 377$). Note that the number of boundary pixels ($336$) is a poor estimate of the circumference, since $336 \neq 377$.

In [ ]:
radius = 60
img = np.zeros((150, 150), dtype=np.uint8)
cv2.circle(img, (75, 75), radius, 255, 1)

contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
pts = contours[0][:, 0, :]

plt.imshow(img, cmap='gray')
plt.title(f'Digitized circle, radius={radius}, {len(pts)} boundary pixels')
plt.axis('off')
plt.show()

A **Freeman chain code** describes a digital curve as a sequence of unit steps in 8 possible directions (0–7, 45° apart). Four of the eight directions are *even* (horizontal/vertical, true length 1) and four are *odd* (diagonal, true length $\sqrt{2}$).

If a curve's chain code has $N_e$ even steps and $N_o$ odd steps, **Freeman's estimate** of its length is simply

$$L_{\text{Freeman}} = N_e + \sqrt{2}\,N_o$$

This sounds right, but it systematically *overestimates* smooth curves: a real circle's boundary, when digitized, alternates through many short zig-zag "staircase" steps that add up to more than the true arc length.

In [ ]:
steps = np.diff(np.vstack([pts, pts[:1]]), axis=0)  # step from each boundary pixel to the next
is_diagonal = (steps[:, 0] != 0) & (steps[:, 1] != 0)

N_o = int(np.sum(is_diagonal))    # odd (diagonal) steps
N_e = int(np.sum(~is_diagonal))   # even (axis-aligned) steps

true_length = 2 * np.pi * radius
L_freeman = N_e + np.sqrt(2) * N_o

print(f'N_e (even/axis steps)     = {N_e}')
print(f'N_o (odd/diagonal steps)  = {N_o}')
print(f'number of boundary pixels = {len(pts):.1f}')
print(f'true circumference        = {true_length:.1f}')
print(f'Freeman estimate          = {L_freeman:.1f}  (error {100*(L_freeman-true_length)/true_length:+.1f}%)')

### A refined weighted estimate

Because Freeman's estimate is systematically biased, several authors proposed replacing the exact weights $(1, \sqrt{2})$ with weights $(a, b)$ tuned to minimize the average error over many digitized curves — an idea sometimes associated with Kimura's chain-code length correction. Commonly cited values (they vary slightly by source) are approximately $a \approx 0.948$ and $b \approx 1.343$:

$$L_{\text{corrected}} = a\,N_e + b\,N_o$$

In [ ]:
a, b = 0.948, 1.343
L_corrected = a * N_e + b * N_o

print(f'corrected estimate         = {L_corrected:.1f}  (error {100*(L_corrected-true_length)/true_length:+.1f}%)')

labels = ['Freeman\n(1, sqrt(2))', 'Corrected\n(0.948, 1.343)']
estimates = [L_freeman, L_corrected]

plt.bar(labels, estimates, color=['#3498db', '#2ecc71'])
plt.axhline(true_length, color='red', linestyle='--', label=f'true = {true_length:.1f}')
plt.ylabel('estimated length')
plt.legend()
plt.title('Chain-code length estimates vs. true circumference')
plt.show()

The corrected weights bring the estimate much closer to the true circumference, without needing anything more than a count of even and odd steps — useful when you want a cheap length estimate directly from a chain code, without ever computing a square root per step.

## Part 3: Chamfer distance transform

A **distance transform** labels every pixel of a binary image with its distance to the nearest foreground (or background) pixel — useful for skeletonization, shape matching, and path planning. Computing it *exactly* with Euclidean distance requires comparing every pixel against every foreground pixel (slow), or a more careful algorithm.

The **chamfer distance transform** is a fast approximation: instead of a single global search, it propagates local distances across the image in just two raster-scan passes (top-left to bottom-right, then bottom-right to top-left), adding a small fixed cost for each step to a neighbor. Using integer weights $a=3$ for axis-aligned neighbors and $b=4$ for diagonal neighbors (then dividing by 3 at the end) gives the classic **3-4 chamfer distance**, a good, cheap approximation to Euclidean distance — the same idea as the Freeman/corrected chain-code weights from Part 2, but applied to every pixel instead of just a boundary.

In [ ]:
def chamfer_distance_transform(binary, a=3, b=4):
    """3-4 chamfer distance transform: distance from each background pixel
    to the nearest foreground (nonzero) pixel, via two raster-scan passes."""
    INF = np.iinfo(np.int32).max // 2
    h, w = binary.shape
    dist = np.where(binary > 0, 0, INF).astype(np.int32)

    # forward pass: top-left to bottom-right, look at neighbors already visited
    for y in range(h):
        for x in range(w):
            if dist[y, x] == 0:
                continue
            best = dist[y, x]
            if x > 0:
                best = min(best, dist[y, x - 1] + a)
            if y > 0:
                best = min(best, dist[y - 1, x] + a)
            if x > 0 and y > 0:
                best = min(best, dist[y - 1, x - 1] + b)
            if x < w - 1 and y > 0:
                best = min(best, dist[y - 1, x + 1] + b)
            dist[y, x] = best

    # backward pass: bottom-right to top-left
    for y in range(h - 1, -1, -1):
        for x in range(w - 1, -1, -1):
            if dist[y, x] == 0:
                continue
            best = dist[y, x]
            if x < w - 1:
                best = min(best, dist[y, x + 1] + a)
            if y < h - 1:
                best = min(best, dist[y + 1, x] + a)
            if x < w - 1 and y < h - 1:
                best = min(best, dist[y + 1, x + 1] + b)
            if x > 0 and y < h - 1:
                best = min(best, dist[y + 1, x - 1] + b)
            dist[y, x] = best

    return dist / a  # rescale so axis-aligned steps count as ~1, like true distance


shape = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(shape, (50, 50), 30, 255, -1)

chamfer = chamfer_distance_transform(shape)

plt.imshow(chamfer, cmap='viridis')
plt.contour(chamfer, levels=10, colors='white', linewidths=0.5)
plt.colorbar(label='distance to nearest foreground pixel')
plt.title('3-4 chamfer distance transform')
plt.axis('off')
plt.show()

### How close is the approximation?

OpenCV's `cv2.distanceTransform` with `DIST_L2` computes the (near-)exact Euclidean distance transform. We compare it to our two-pass chamfer approximation on the same image.

In [ ]:
background_is_nonzero = (255 - shape)  # cv2.distanceTransform measures distance to nearest zero pixel
exact = cv2.distanceTransform(background_is_nonzero, cv2.DIST_L2, 5)

error = chamfer - exact
print(f'max error  = {np.abs(error).max():.3f} pixels')
print(f'mean error = {np.abs(error).mean():.3f} pixels')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, field, title in zip(
    axes,
    [chamfer, exact, error],
    ['Chamfer (3-4) approx.', 'Exact (cv2.distanceTransform)', 'Difference'],
):
    im = ax.imshow(field, cmap='viridis' if title != 'Difference' else 'coolwarm')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

The chamfer transform matches the exact Euclidean distance transform closely near the axes and diagonals (where its integer weights are exact or near-exact) and drifts by a small amount elsewhere — the same fundamental trade-off as the Freeman vs. corrected chain-code estimates in Part 2, just extended from a single curve to every pixel in the image, computed in two fast linear passes instead of an expensive nearest-neighbor search.

### Exercise

1. Repeat the circle experiment for a few different radii (e.g. 10, 20, 40, 80). Does the Freeman estimate's percent error stay roughly constant, or does it shrink as the circle gets bigger? Why might that be?
2. The chessboard distance $D_8$ is the exact shortest path length when diagonal moves are allowed and cost 1. Can you construct a modified chessboard-like metric, using a diagonal cost of $\sqrt{2}$ instead of 1, that behaves like a coarse local version of the chain-code length estimators above? Compare it to true Euclidean distance for a few points.
3. Try the chamfer weights $(a,b) = (1,1)$ (equivalent to chessboard distance) and $(a,b)=(1,\sqrt{2})$ (locally exact weights, no integer rescaling) in `chamfer_distance_transform`. How does the max/mean error compared to the exact transform change for each?
4. `cv2.distanceTransform` also accepts a 3x3 mask (pass `3` instead of `5`), which is itself an efficient two-pass chamfer-style approximation internally. Compare its output and error to both your hand-rolled chamfer transform and the 5x5 result used above.